# Validação a 10% — NYC Yellow Taxi 2023–2024

Aqui conferimos M1, M2, M3, O1 e O2 com 10% dos dados. O código dos cálculos está em `projeto1.py`; este notebook importa as funções e verifica os resultados.

Ordem: **preparar → M1 → M2 → M3 → O1 → O2 → estabilidade → fechar**. Para repetir, usar um kernel novo e seguir as células de cima para baixo. Se alterarmos uma função, recarregar a preparação e repetir as verificações afetadas.

As saídas guardadas são da execução anterior, já concluída. Nesta organização preservámos os resultados sem voltar a correr Spark. Os números `[ ]` mostram a execução histórica, por isso podem ter intervalos; os títulos dão a ordem de leitura.

As comparações de desempenho estão no [notebook de experiências](experiências%20projeto1.ipynb). O arquivo exploratório desse notebook conserva os tempos e o código que antes estavam aqui.


## 1. Preparar o ambiente

### 1.1 Ligar ao Spark

`DOCKER = True` usa o nosso cluster. `FRACTION = 0.1` seleciona uma amostra de 10%.


In [1]:
import os, sys, importlib

DOCKER = True          # True dentro do Jupyter do compose.project.yml

if DOCKER:
    master, project = "spark://spark-master:7077", "/project"
else:
    os.environ["PYSPARK_PYTHON"] = sys.executable
    os.environ["SPARK_LOCAL_IP"] = "127.0.0.1"
    master, project = "local[*]", os.path.abspath("..") if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd()

dataFolder = os.path.join(project, "data", "raw")
outFolder  = os.path.join(project, "data", "output-notebook")
YEARS      = [2023, 2024]
FRACTION   = 0.1        # começar com 10%; 1.0 para os números finais

sys.path.append(project)
import projeto1 as pl
importlib.reload(pl)

from pyspark.sql import SparkSession
import pyspark.sql.functions as sf

builder = (SparkSession.builder.master(master).appName("taxi-notebook")
           .config("spark.sql.session.timeZone", "UTC")
           .config("spark.ui.showConsoleProgress", "false"))
if DOCKER:
    builder = (builder.config("spark.driver.host", "jupyter")
                      .config("spark.driver.bindAddress", "0.0.0.0")
                      .config("spark.executor.memory", "1536m"))
else:
    builder = (builder.config("spark.driver.memory", "8g")
                      .config("spark.driver.host", "127.0.0.1")
                      .config("spark.driver.bindAddress", "127.0.0.1"))
spark = builder.getOrCreate()
print("Spark", spark.version, "| master:", spark.sparkContext.master,
      "| parallelism:", spark.sparkContext.defaultParallelism)
print("Spark UI:", spark.sparkContext.uiWebUrl)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


26/11/05 23:52:16 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark 4.2.0 | master: spark://spark-master:7077 | parallelism: 6
Spark UI: http://jupyter:4040


### 1.2 Confirmar os ficheiros

Devem existir 12 ficheiros por ano e a tabela de zonas.


In [2]:
for year in YEARS:
    files = pl.find_trip_files(dataFolder, year)

    expected = {
        f"yellow_tripdata_{year}-{month:02}.parquet"
        for month in range(1, 13)
    }
    found = {os.path.basename(file) for file in files}

    print(f"{year}: {len(files)} ficheiros")
    print("Meses em falta:", sorted(expected - found))
    print("Ficheiros inesperados:", sorted(found - expected))

lookup = os.path.join(dataFolder, "lookup", "taxi_zone_lookup.csv")
print("Tabela de zonas presente:", os.path.isfile(lookup))

2023: 12 ficheiros
Meses em falta: []
Ficheiros inesperados: []
2024: 12 ficheiros
Meses em falta: []
Ficheiros inesperados: []
Tabela de zonas presente: True


### 1.3 Separar validação de desempenho

Aqui calculamos e conferimos resultados. As medições completas e as decisões de desempenho estão no notebook de experiências. A comparação entre os dois métodos de O1 fica na secção 5.5, porque verifica a igualdade dos resultados.


## 2. M1 — ler e limpar

### 2.1 Ler e contar os problemas

Marcamos os problemas antes de remover viagens. Uma viagem pode falhar várias regras.


In [4]:
trips, n_files = pl.read_trips(spark, dataFolder, YEARS, FRACTION)
zones = pl.read_zones(spark, dataFolder)
print(n_files, "ficheiros lidos")

flagged = pl.add_quality_flags(trips)
stats_pd = pl.cleaning_stats(flagged).toPandas()
stats_pd

24 ficheiros lidos


,source_year,total,f_nulls,f_fare,f_distance,f_passengers,f_time,f_location,f_period,f_any,kept
0,2023,3830507,130871,39519,76843,58318,1540,63346,9,322777,3507730
1,2024,4116927,408461,74527,77246,40222,1322,42599,3,588475,3528452


### 2.2 Ver as percentagens

Os critérios podem sobrepor-se, por isso não somamos as percentagens.


In [5]:
# percentagens por critério (para o relatório)
pct = stats_pd.copy()
for f in pl.QUALITY_FLAGS:
    pct[f] = (100 * pct[f] / pct["total"]).round(2)
pct

,source_year,total,f_nulls,f_fare,f_distance,f_passengers,f_time,f_location,f_period,f_any,kept
0,2023,3830507,3.42,1.03,2.01,1.52,0.04,1.65,0.0,8.43,3507730
1,2024,4116927,9.92,1.81,1.88,0.98,0.03,1.03,0.0,14.29,3528452


### 2.3 Limpar e preencher a cache

O count executa o cálculo e preenche a cache.


In [6]:
clean = pl.clean_trips(flagged).cache()
n_clean = clean.count()
print("viagens válidas:", n_clean)

viagens válidas: 7036182


### 2.4 Confirmar as contagens por ano

As contagens em clean devem coincidir com kept nesta execução.


In [7]:
clean.groupBy("source_year").count().orderBy("source_year").show()
stats_pd[["source_year", "total", "f_any", "kept"]]

+-----------+-------+
|source_year|  count|
+-----------+-------+
|       2023|3507730|
|       2024|3528452|
+-----------+-------+



,source_year,total,f_any,kept
0,2023,3830507,322777,3507730
1,2024,4116927,588475,3528452


### 2.5 Reaplicar as regras

Esperamos zero viagens inválidas. Este teste verifica as regras implementadas.


In [8]:
remaining_invalid = (
    pl.add_quality_flags(clean)
    .filter(sf.col("f_any"))
    .count()
)

print("Viagens inválidas que ficaram:", remaining_invalid)

Viagens inválidas que ficaram: 0


### 2.6 Ler o plano físico

Procurar Scan parquet, ReadSchema e Filter. InMemoryTableScan indica leitura da cache. O plano explica a execução; a melhoria de desempenho precisa de medições.


In [9]:
pl.clean_trips(flagged).explain("formatted")

== Physical Plan ==
InMemoryTableScan (1)
   +- InMemoryRelation (2)
         +- * Project (78)
            +- * Filter (77)
               +- * Sample (76)
                  +- Union (75)
                     :- * Project (5)
                     :  +- * ColumnarToRow (4)
                     :     +- Scan parquet  (3)
                     :- * Project (8)
                     :  +- * ColumnarToRow (7)
                     :     +- Scan parquet  (6)
                     :- * Project (11)
                     :  +- * ColumnarToRow (10)
                     :     +- Scan parquet  (9)
                     :- * Project (14)
                     :  +- * ColumnarToRow (13)
                     :     +- Scan parquet  (12)
                     :- * Project (17)
                     :  +- * ColumnarToRow (16)
                     :     +- Scan parquet  (15)
                     :- * Project (20)
                     :  +- * ColumnarToRow (19)
                     :     +- Scan parquet  (18)
  

## 3. M2 — procura por borough, dia e hora

### 3.1 Juntar as zonas de partida

O join acrescenta o borough e a zona. A contagem deve manter-se.


In [10]:
trips_pu = pl.add_pickup_location(clean, zones)

n_before_join = clean.count()
n_after_join = trips_pu.count()

print("Antes do join:", n_before_join)
print("Depois do join:", n_after_join)
print("Diferença:", n_after_join - n_before_join)

Antes do join: 7036182
Depois do join: 7036182
Diferença: 0


### 3.2 Calcular e conferir o total

Cada linha representa um grupo. A soma de trip_count deve coincidir com as viagens usadas.


In [11]:
m2 = pl.m2_demand(trips_pu)

m2.orderBy(
    "pickup_borough", "day_of_week", "hour_of_day"
).show(10, truncate=False)

total_m2 = m2.agg(sf.sum("trip_count").alias("total")).first()["total"]
print("Soma das viagens em M2:", total_m2)
print("Viagens utilizadas:", n_after_join)
print("Contagens coincidem:", total_m2 == n_after_join)


+--------------+-----------+-----------+----------+----------------+--------------------------+---------------------+
|pickup_borough|day_of_week|hour_of_day|trip_count|mean_fare_amount|mean_trip_duration_minutes|mean_trip_distance_km|
+--------------+-----------+-----------+----------+----------------+--------------------------+---------------------+
|Bronx         |1          |0          |11        |22.28           |12.68                     |5.565                |
|Bronx         |1          |1          |13        |20.95           |20.55                     |5.989                |
|Bronx         |1          |2          |16        |39.03           |13.00                     |7.655                |
|Bronx         |1          |3          |10        |25.31           |9.45                      |4.603                |
|Bronx         |1          |4          |12        |34.58           |20.49                     |12.9                 |
|Bronx         |1          |5          |12        |28.08

Soma das viagens em M2: 7036182
Viagens utilizadas: 7036182
Contagens coincidem: True


### 3.3 Conferir um grupo diretamente

Bronx, domingo, entre as 00h e a 01h. Usamos o mesmo tipo decimal de M2 na média monetária.


In [12]:
# Exemplo: Bronx, domingo, entre as 00h e a 01h
group = trips_pu.filter(
    (sf.col("pickup_borough") == "Bronx")
    & (sf.col("day_of_week") == 1)
    & (sf.col("hour_of_day") == 0)
)

print("Cálculo direto nas viagens:")
group.agg(
    sf.count("*").alias("trip_count"),
    sf.round(sf.avg(sf.col("fare_amount").cast("decimal(20, 6)")), 2).alias("mean_fare_amount"),
    sf.round(sf.avg((sf.unix_timestamp("tpep_dropoff_datetime") - sf.unix_timestamp("tpep_pickup_datetime")).cast("decimal(20, 0)")) / 60, 2)
      .alias("mean_trip_duration_minutes"),
    sf.round(sf.avg("trip_distance_km"), 3)
      .alias("mean_trip_distance_km")
).show()

print("Resultado produzido por M2:")
m2.filter(
    (sf.col("pickup_borough") == "Bronx")
    & (sf.col("day_of_week") == 1)
    & (sf.col("hour_of_day") == 0)
).show(truncate=False)

Cálculo direto nas viagens:


+----------+----------------+--------------------------+---------------------+
|trip_count|mean_fare_amount|mean_trip_duration_minutes|mean_trip_distance_km|
+----------+----------------+--------------------------+---------------------+
|        11|           22.28|                     12.68|                5.565|
+----------+----------------+--------------------------+---------------------+

Resultado produzido por M2:


+--------------+-----------+-----------+----------+----------------+--------------------------+---------------------+
|pickup_borough|day_of_week|hour_of_day|trip_count|mean_fare_amount|mean_trip_duration_minutes|mean_trip_distance_km|
+--------------+-----------+-----------+----------+----------------+--------------------------+---------------------+
|Bronx         |1          |0          |11        |22.28           |12.68                     |5.565                |
+--------------+-----------+-----------+----------+----------------+--------------------------+---------------------+



### 3.4 Conferir o arredondamento monetário

Conferimos o grupo usado no diagnóstico monetário. A função atual calcula a média em decimal.


In [13]:
m2 = pl.m2_demand(trips_pu)

m2.filter(
    (sf.col("pickup_borough") == "Brooklyn")
    & (sf.col("day_of_week") == 2)
    & (sf.col("hour_of_day") == 23)
).show(truncate=False)

+--------------+-----------+-----------+----------+----------------+--------------------------+---------------------+
|pickup_borough|day_of_week|hour_of_day|trip_count|mean_fare_amount|mean_trip_duration_minutes|mean_trip_distance_km|
+--------------+-----------+-----------+----------+----------------+--------------------------+---------------------+
|Brooklyn      |2          |23         |100       |23.06           |14.84                     |6.544                |
+--------------+-----------+-----------+----------+----------------+--------------------------+---------------------+



### 3.5 Escrever, ler e comparar

Fixamos apenas o pequeno resultado agregado para testar a escrita sem recalcular as médias. Esperamos True.


In [14]:
m2_path = os.path.join(outFolder, "validacao_m2")

# Calcular uma vez e fixar os valores para este teste
rows = m2.collect()
m2_fixed = spark.createDataFrame(rows, schema=m2.schema)

# Escrever exatamente esses valores
m2_fixed.write.mode("overwrite") \
    .partitionBy("pickup_borough") \
    .parquet(m2_path)

# Ler e comparar, sem recalcular a agregação
saved = spark.read.parquet(m2_path).select(m2_fixed.columns)

same = (
    m2_fixed.exceptAll(saved).count() == 0
    and saved.exceptAll(m2_fixed).count() == 0
)

print("Escrita preservou os valores:", same)

Escrita preservou os valores: True


## 4. M3 — resultados por zona e mês

### 4.1 Calcular e conferir as linhas

Esperamos uma linha por zona e mês, incluindo os meses sem viagens.


In [15]:
m3 = pl.m3_zone_month(spark, trips_pu, zones, YEARS)

n_zones = zones.filter(
    sf.col("LocationID").between(1, 263)
).select("LocationID").distinct().count()

expected_rows = n_zones * 12 * len(YEARS)

print("Linhas esperadas:", expected_rows)
print("Linhas obtidas:", m3.count())

Linhas esperadas: 6312


Linhas obtidas: 6312


### 4.2 Conferir chaves e contagens

Sem combinações repetidas. A soma das viagens deve coincidir com o total usado.


In [16]:
duplicates = (
    m3.groupBy("PULocationID", "pickup_year", "pickup_month")
    .count()
    .filter(sf.col("count") > 1)
    .count()
)

total_m3 = m3.agg(sf.sum("trip_count")).first()[0]

print("Combinações zona/mês repetidas:", duplicates)
print("Soma das viagens em M3:", total_m3)
print("Viagens utilizadas:", n_after_join)
print("Contagens coincidem:", total_m3 == n_after_join)

Combinações zona/mês repetidas: 0
Soma das viagens em M3: 7036182
Viagens utilizadas: 7036182
Contagens coincidem: True


### 4.3 Conferir médias e desvios-padrão

Comparamos a zona 186, em abril de 2023, com o cálculo direto nas viagens.


In [17]:
group = trips_pu.filter(
    (sf.col("PULocationID") == 186)
    & (sf.col("pickup_year") == 2023)
    & (sf.col("pickup_month") == 4)
)

print("Cálculo direto nas viagens:")
group.agg(
    sf.count("*").alias("trip_count"),
    sf.round(sf.avg(sf.col("fare_amount").cast("decimal(20, 6)")), 2).alias("mean_fare_amount"),
    sf.round(sf.stddev("fare_amount"), 2).alias("std_fare_amount"),
    sf.round(sf.avg("trip_distance_km"), 3).alias("mean_trip_distance_km"),
    sf.round(sf.stddev("trip_distance_km"), 3).alias("std_trip_distance_km")
).show(truncate=False)

print("Resultado de M3:")
m3.filter(
    (sf.col("PULocationID") == 186)
    & (sf.col("pickup_year") == 2023)
    & (sf.col("pickup_month") == 4)
).select(
    "trip_count",
    "mean_fare_amount",
    "std_fare_amount",
    "mean_trip_distance_km",
    "std_trip_distance_km"
).show(truncate=False)

Cálculo direto nas viagens:


+----------+----------------+---------------+---------------------+--------------------+
|trip_count|mean_fare_amount|std_fare_amount|mean_trip_distance_km|std_trip_distance_km|
+----------+----------------+---------------+---------------------+--------------------+
|10089     |17.01           |11.28          |3.946                |4.073               |
+----------+----------------+---------------+---------------------+--------------------+

Resultado de M3:


+----------+----------------+---------------+---------------------+--------------------+
|trip_count|mean_fare_amount|std_fare_amount|mean_trip_distance_km|std_trip_distance_km|
+----------+----------------+---------------+---------------------+--------------------+
|10089     |17.01           |11.28          |3.946                |4.073               |
+----------+----------------+---------------+---------------------+--------------------+



### 4.4 Ver as janelas e as anomalias

A anomalia usa os três meses anteriores. Nos primeiros três meses falta histórico completo.


In [18]:
m3.filter(
    sf.col("PULocationID") == 186
).select(
    "pickup_year",
    "pickup_month",
    "trip_count",
    "rolling_mean_3m",
    "rolling_std_3m",
    "volume_anomaly"
).orderBy(
    "pickup_year", "pickup_month"
).show(15, truncate=False)

+-----------+------------+----------+---------------+--------------+--------------+
|pickup_year|pickup_month|trip_count|rolling_mean_3m|rolling_std_3m|volume_anomaly|
+-----------+------------+----------+---------------+--------------+--------------+
|2023       |1           |10272     |NULL           |NULL          |NULL          |
|2023       |2           |10053     |10272.0        |NULL          |NULL          |
|2023       |3           |10936     |10162.5        |154.9         |NULL          |
|2023       |4           |10089     |10420.3        |459.8         |false         |
|2023       |5           |10688     |10359.3        |499.7         |false         |
|2023       |6           |10458     |10571.0        |435.5         |false         |
|2023       |7           |10320     |10411.7        |302.2         |false         |
|2023       |8           |10150     |10488.7        |185.9         |true          |
|2023       |9           |9091      |10309.3        |154.3         |true    

### 4.5 Conferir a soma das participações

Devem somar aproximadamente 1 por borough e mês com volume. A tolerância aceita o arredondamento a cinco casas.


In [19]:
shares = (
    m3.groupBy("pickup_borough", "pickup_year", "pickup_month")
    .agg(
        sf.sum("share_3m").alias("share_sum"),
        sf.count("share_3m").alias("zones_with_share")
    )
)

shares.orderBy(
    "pickup_borough", "pickup_year", "pickup_month"
).show(15, truncate=False)

problems = shares.filter(
    (sf.col("zones_with_share") > 0)
    & (sf.abs(sf.col("share_sum") - 1) > 0.002)
)

print("Grupos fora da tolerância:", problems.count())

+--------------+-----------+------------+------------------+----------------+
|pickup_borough|pickup_year|pickup_month|share_sum         |zones_with_share|
+--------------+-----------+------------+------------------+----------------+
|Bronx         |2023       |1           |0.99999           |43              |
|Bronx         |2023       |2           |0.9999899999999998|43              |
|Bronx         |2023       |3           |0.9999899999999999|43              |
|Bronx         |2023       |4           |0.9999899999999998|43              |
|Bronx         |2023       |5           |1.00002           |43              |
|Bronx         |2023       |6           |0.9999899999999999|43              |
|Bronx         |2023       |7           |1.0               |43              |
|Bronx         |2023       |8           |1.0               |43              |
|Bronx         |2023       |9           |0.9999999999999998|43              |
|Bronx         |2023       |10          |1.0000299999999998|43  

Grupos fora da tolerância: 0


### 4.6 Conferir uma participação diretamente

Para abril de 2023 usamos fevereiro, março e abril: viagens da zona a dividir pelas do borough.


In [20]:
# Abril usa fevereiro, março e abril
period = trips_pu.filter(
    (sf.col("pickup_year") == 2023)
    & sf.col("pickup_month").isin(2, 3, 4)
)

zone_trips = period.filter(sf.col("PULocationID") == 186).count()
borough_trips = period.filter(
    sf.col("pickup_borough") == "Manhattan"
).count()

print("Participação calculada diretamente:",
      round(zone_trips / borough_trips, 5))

m3.filter(
    (sf.col("PULocationID") == 186)
    & (sf.col("pickup_year") == 2023)
    & (sf.col("pickup_month") == 4)
).select("share_3m").show()

Participação calculada diretamente: 0.03886


+--------+
|share_3m|
+--------+
| 0.03886|
+--------+



### 4.7 Ver a classificação das zonas

As zonas com mais viagens aparecem primeiro dentro do mesmo borough e mês.


In [21]:
m3.filter(
    (sf.col("pickup_borough") == "Manhattan")
    & (sf.col("pickup_year") == 2023)
    & (sf.col("pickup_month") == 4)
).select(
    "PULocationID",
    "pickup_zone",
    "trip_count",
    "rank_in_borough"
).orderBy(
    sf.desc("trip_count"), "PULocationID"
).show(10, truncate=False)

+------------+----------------------------+----------+---------------+
|PULocationID|pickup_zone                 |trip_count|rank_in_borough|
+------------+----------------------------+----------+---------------+
|237         |Upper East Side South       |14693     |1              |
|161         |Midtown Center              |14288     |2              |
|236         |Upper East Side North       |13085     |3              |
|162         |Midtown East                |10750     |4              |
|142         |Lincoln Square East         |10612     |5              |
|230         |Times Sq/Theatre District   |10251     |6              |
|186         |Penn Station/Madison Sq West|10089     |7              |
|170         |Murray Hill                 |9223      |8              |
|163         |Midtown North               |9034      |9              |
|239         |Upper West Side South       |8594      |10             |
+------------+----------------------------+----------+---------------+
only s

### 4.8 Resumir as anomalias

Contamos combinações zona/mês com anomalia, não viagens individuais.


In [22]:
m3_pd = pl.m3_zone_month(spark, trips_pu, zones, YEARS).toPandas()
print(len(m3_pd), "linhas (zonas x meses)")
m3_pd[m3_pd["volume_anomaly"] == True].groupby("pickup_borough").size()

6312 linhas (zonas x meses)


pickup_borough
Bronx            278
Brooklyn         467
EWR                9
Manhattan        502
Queens           464
Staten Island     32
dtype: int64

### 4.9 Escrever, ler e comparar

Fixamos o resultado agregado e confirmamos que a escrita por borough preserva os valores.


In [23]:
m3_fixed = spark.createDataFrame(m3.collect(), schema=m3.schema)
m3_path = os.path.join(outFolder, "validacao_m3")

m3_fixed.write.mode("overwrite") \
    .partitionBy("pickup_borough") \
    .parquet(m3_path)

m3_saved = spark.read.parquet(m3_path).select(m3_fixed.columns)

same = (
    m3_fixed.exceptAll(m3_saved).count() == 0
    and m3_saved.exceptAll(m3_fixed).count() == 0
)

print("Pastas por borough:")
print(sorted(
    name for name in os.listdir(m3_path)
    if name.startswith("pickup_borough=")
))

print("Linhas antes:", m3_fixed.count())
print("Linhas depois:", m3_saved.count())
print("Conteúdo preservado:", same)

Pastas por borough:
['pickup_borough=Bronx', 'pickup_borough=Brooklyn', 'pickup_borough=EWR', 'pickup_borough=Manhattan', 'pickup_borough=Queens', 'pickup_borough=Staten Island']
Linhas antes: 6312


Linhas depois: 6312
Conteúdo preservado: True


## 5. O1 — anomalias nas tarifas

### 5.1 Calcular as taxas

Uma viagem é sinalizada se a tarifa por km ultrapassar média + 3 × desvio-padrão da mesma zona e hora. Depois resumimos por borough, pagamento e tipo de dia. É um sinal estatístico, não prova de erro.


In [24]:
o1_g = pl.o1_fare_anomalies(trips_pu, "groupby").toPandas()
o1_g


,day_type,payment_type,pickup_borough,trips,anomalies,anomaly_rate_pct
0,weekday,1,Bronx,8581,200,2.331
1,weekend,1,Bronx,1647,49,2.975
2,weekday,2,Bronx,830,17,2.048
3,weekend,2,Bronx,394,6,1.523
4,weekday,3,Bronx,35,6,17.143
5,weekend,3,Bronx,16,0,0.000
6,weekday,4,Bronx,45,0,0.000
7,weekend,4,Bronx,31,1,3.226
8,weekday,1,Brooklyn,30865,562,1.821
9,weekend,1,Brooklyn,10182,156,1.532


### 5.2 Conferir contagens e limites

A soma das viagens deve coincidir com o total usado. As anomalias ficam entre zero e o número de viagens; as taxas, entre 0% e 100%.


In [25]:
# Confirmar que O1 contabiliza todas as viagens.
total_o1 = int(o1_g["trips"].sum())

invalid_groups = o1_g[
    (o1_g["anomalies"] < 0)
    | (o1_g["anomalies"] > o1_g["trips"])
    | (~o1_g["anomaly_rate_pct"].between(0, 100))
]

print("Viagens em O1:", total_o1)
print("Viagens utilizadas:", n_after_join)
print("Contagens coincidem:", total_o1 == n_after_join)
print("Grupos com valores inválidos:", len(invalid_groups))

Viagens em O1: 7036182
Viagens utilizadas: 7036182
Contagens coincidem: True
Grupos com valores inválidos: 0


### 5.3 Comparar dias úteis e fins de semana

A tabela mostra percentagens por borough e pagamento. Um NaN pode indicar que não existem viagens nessa combinação, não uma taxa de zero.


In [26]:
# taxa de anomalias: dia útil vs fim de semana, por borough
o1_g.pivot_table(index=["pickup_borough", "payment_type"], columns="day_type",
                 values="anomaly_rate_pct").round(2)

day_type                     weekday  weekend
pickup_borough payment_type                  
Bronx          1                2.33     2.98
               2                2.05     1.52
               3               17.14     0.00
               4                0.00     3.23
Brooklyn       1                1.82     1.53
               2                2.05     1.61
               3               10.53     9.09
               4                7.10     8.11
EWR            1                2.83     3.28
               2                0.00     0.00
               3                0.00     0.00
               4                0.00     0.00
Manhattan      1                0.06     0.07
               2                0.29     0.33
               3                2.18     2.79
               4                2.23     2.50
Queens         1                0.33     0.39
               2                0.43     0.44
               3                3.55     2.11
               4                1.77     1.98
Staten Island  1                0.94     0.00
               2                0.00     0.00
               4                0.00      NaN

### 5.4 Conferir um grupo diretamente

Usamos a zona 186, à hora 0. Calculamos o limite e contamos as anomalias diretamente; a função O1 deve dar a mesma contagem. A viagem apresentada deve ultrapassar o limite.


In [27]:
# Conferir O1 num único grupo: zona 186, à hora 0.
example = trips_pu.filter(
    (sf.col("PULocationID") == 186)
    & (sf.col("hour_of_day") == 0)
).withColumn(
    "fare_per_km",
    sf.col("fare_amount") / sf.col("trip_distance_km")
)

# Calcular a referência diretamente neste grupo.
reference = example.agg(
    sf.avg("fare_per_km").alias("mean"),
    sf.stddev("fare_per_km").alias("std")
).first()

limit = reference["mean"] + 3 * reference["std"]

direct_count = example.filter(
    sf.col("fare_per_km") > limit
).count()

# A função deve identificar o mesmo número de anomalias.
o1_count = (
    pl.o1_fare_anomalies(example, "groupby")
    .agg(sf.sum("anomalies"))
    .first()[0]
)

print("Limite de tarifa por km:", limit)
print("Anomalias no cálculo direto:", direct_count)
print("Anomalias na função O1:", o1_count)
print("Contagens coincidem:", direct_count == o1_count)

# Mostrar uma viagem sinalizada e o limite que ultrapassou.
example.filter(
    sf.col("fare_per_km") > limit
).select(
    "fare_amount", "trip_distance_km", "fare_per_km",
    sf.lit(limit).alias("anomaly_limit")
).show(1, truncate=False)

Limite de tarifa por km: 265.91266986287775
Anomalias no cálculo direto: 4
Anomalias na função O1: 4
Contagens coincidem: True


+-----------+----------------+-----------------+------------------+
|fare_amount|trip_distance_km|fare_per_km      |anomaly_limit     |
+-----------+----------------+-----------------+------------------+
|110.0      |0.01609344      |6835.083114610674|265.91266986287775|
+-----------+----------------+-----------------+------------------+
only showing top 1 row


### 5.5 Confirmar a igualdade entre janela e groupBy

O resultado por groupBy já foi calculado em 5.1. Agora calculamos a alternativa com janela e alinhamos as chaves para comparar contagens, anomalias e taxas. Esperamos True. Este é um teste de correção; os tempos são analisados no notebook de experiências.


In [ ]:
# Calcular a alternativa para conferir os mesmos resultados.
o1_w = pl.o1_fare_anomalies(trips_pu, "window").toPandas()


In [39]:
# Comparar os resultados dos dois métodos, pela mesma ordem.
keys = ["pickup_borough", "payment_type", "day_type"]

window_result = o1_w.sort_values(keys).reset_index(drop=True)
groupby_result = o1_g.sort_values(keys).reset_index(drop=True)

print(
    "Resultados iguais:",
    window_result.equals(groupby_result)
)

Resultados iguais: True


Validação com 10%: as contagens, os limites, o exemplo direto e a comparação dos métodos passaram na execução guardada. As medições controladas de O1 estão no notebook de experiências.


## 6. O2 — rotas

### 6.1 Calcular os resultados e ver exemplos

Criamos as rotas mensais, as inversões de tendência e o top de 20 por borough de partida. Mostramos só três exemplos por borough.


In [28]:
route_month, reversal, top = pl.o2_routes(clean, zones)
top_pd = top.toPandas()
top_pd.groupby("pickup_borough").head(3)

,PULocationID,DOLocationID,pickup_borough,pickup_zone,dropoff_borough,dropoff_zone,total_trips,route_rank,years_with_reversal
0,168,74,Bronx,Mott Haven/Port Morris,Manhattan,East Harlem North,59,1,1
1,168,168,Bronx,Mott Haven/Port Morris,Bronx,Mott Haven/Port Morris,50,2,1
2,247,42,Bronx,West Concourse,Manhattan,Central Harlem North,47,3,2
20,181,181,Brooklyn,Park Slope,Brooklyn,Park Slope,192,1,1
21,255,112,Brooklyn,Williamsburg (North Side),Brooklyn,Greenpoint,184,2,2
22,52,33,Brooklyn,Cobble Hill,Brooklyn,Brooklyn Heights,180,3,1
40,1,1,EWR,Newark Airport,EWR,Newark Airport,168,1,2
41,1,132,EWR,Newark Airport,Queens,JFK Airport,3,2,0
42,1,239,EWR,Newark Airport,Manhattan,Upper West Side South,3,3,0
60,237,236,Manhattan,Upper East Side South,Manhattan,Upper East Side North,51679,1,2


### 6.2 Conferir o total e as chaves

A soma de trip_count deve coincidir com as viagens limpas. Cada rota deve ter uma única linha por ano e mês.


In [29]:
# Confirmar o total de viagens e a chave de cada rota mensal.
total_o2 = route_month.agg(sf.sum("trip_count")).first()[0]

duplicates = (
    route_month
    .groupBy(
        "PULocationID", "DOLocationID",
        "pickup_year", "pickup_month"
    )
    .count()
    .filter(sf.col("count") > 1)
    .count()
)

print("Viagens em O2:", total_o2)
print("Viagens utilizadas:", n_clean)
print("Contagens coincidem:", total_o2 == n_clean)
print("Rotas mensais repetidas:", duplicates)

Viagens em O2: 7036182
Viagens utilizadas: 7036182
Contagens coincidem: True
Rotas mensais repetidas: 0


### 6.3 Conferir as participações

route_share é a fração das viagens que saem da mesma zona, no mesmo ano e mês. As participações devem somar 1, com tolerância de 0,000001.


In [30]:
# As participações das rotas de cada origem devem somar 1.
shares_o2 = (
    route_month
    .groupBy("PULocationID", "pickup_year", "pickup_month")
    .agg(sf.sum("route_share").alias("share_sum"))
)

invalid_shares = shares_o2.filter(
    sf.col("share_sum").isNull()
    | (sf.abs(sf.col("share_sum") - 1) > 0.000001)
)

print("Grupos com participação inválida:", invalid_shares.count())

Grupos com participação inválida: 0


### 6.4 Conferir as posições do top

Esperamos até 20 rotas por borough, com posições distintas a começar em 1. Isto confere a estrutura; o teste seguinte confere o conteúdo.


In [31]:
# Conferir o número de rotas e as posições por borough.
top_check = (
    top.groupBy("pickup_borough")
    .agg(
        sf.count("*").alias("routes"),
        sf.min("route_rank").alias("first_rank"),
        sf.max("route_rank").alias("last_rank"),
        sf.countDistinct("route_rank").alias("distinct_ranks")
    )
)

top_check.orderBy("pickup_borough").show(truncate=False)

+--------------+------+----------+---------+--------------+
|pickup_borough|routes|first_rank|last_rank|distinct_ranks|
+--------------+------+----------+---------+--------------+
|Bronx         |20    |1         |20       |20            |
|Brooklyn      |20    |1         |20       |20            |
|EWR           |20    |1         |20       |20            |
|Manhattan     |20    |1         |20       |20            |
|Queens        |20    |1         |20       |20            |
|Staten Island |20    |1         |20       |20            |
+--------------+------+----------+---------+--------------+



### 6.5 Conferir o top com as viagens limpas

Voltamos a contar diretamente as rotas do Bronx. Comparamos as 20 primeiras, os totais e a ordem. Nos empates, usamos os identificadores de origem e destino.


In [32]:
# Calcular diretamente as 20 rotas com mais viagens do Bronx.
expected_top = (
    clean
    .join(
        zones.select(
            sf.col("LocationID").alias("PULocationID"),
            sf.col("Borough").alias("pickup_borough")
        ),
        on="PULocationID"
    )
    .filter(sf.col("pickup_borough") == "Bronx")
    .groupBy("PULocationID", "DOLocationID")
    .agg(sf.count("*").alias("total_trips"))
    .orderBy(
        sf.desc("total_trips"),
        "PULocationID",
        "DOLocationID"
    )
    .limit(20)
    .collect()
)

# Comparar as rotas, os totais e a ordem com o resultado de O2.
actual_top = (
    top
    .filter(sf.col("pickup_borough") == "Bronx")
    .orderBy("route_rank")
    .select("PULocationID", "DOLocationID", "total_trips")
    .collect()
)

print("Top do Bronx coincide:", expected_top == actual_top)

Top do Bronx coincide: True


### 6.6 Ver um exemplo de inversão

Na nossa definição, a participação muda num sentido entre janeiro e março e no sentido contrário entre julho e setembro. Os dois deltas devem ter sinais opostos. Mês sem viagens na rota é tratado como participação zero no código.


In [33]:
# Ver um exemplo de inversão e as participações usadas.
reversal.filter(
    sf.col("q1_q3_reversal")
).select(
    "PULocationID", "DOLocationID", "pickup_year",
    "s_jan", "s_mar", "s_jul", "s_sep",
    "delta_q1", "delta_q3", "q1_q3_reversal"
).orderBy(
    "pickup_year", "PULocationID", "DOLocationID"
).show(1, truncate=False)

+------------+------------+-----------+-----+-----+------+-----+--------------------+--------+--------------+
|PULocationID|DOLocationID|pickup_year|s_jan|s_mar|s_jul |s_sep|delta_q1            |delta_q3|q1_q3_reversal|
+------------+------------+-----------+-----+-----+------+-----+--------------------+--------+--------------+
|1           |1           |2023       |1.0  |0.9  |0.4375|1.0  |-0.09999999999999998|0.5625  |true          |
+------------+------------+-----------+-----+-----+------+-----+--------------------+--------+--------------+
only showing top 1 row


### 6.7 Conferir os meses do exemplo

Consultamos janeiro, março, julho e setembro da rota 1 → 1, em 2023. As participações devem coincidir com as usadas no cálculo da inversão dessa rota.


In [34]:
# Conferir os meses usados neste exemplo.
route_month.filter(
    (sf.col("PULocationID") == 1)
    & (sf.col("DOLocationID") == 1)
    & (sf.col("pickup_year") == 2023)
    & sf.col("pickup_month").isin(1, 3, 7, 9)
).select(
    "pickup_month", "trip_count", "route_share"
).orderBy("pickup_month").show()

+------------+----------+-----------+
|pickup_month|trip_count|route_share|
+------------+----------+-----------+
|           1|         4|        1.0|
|           3|         9|        0.9|
|           7|         7|     0.4375|
|           9|         4|        1.0|
+------------+----------+-----------+



### 6.8 Escrever, ler e comparar

Guardamos route_month, reversal e top em pastas separadas. Depois alinhamos as colunas e comparamos nos dois sentidos. Esperamos True nos três resultados.


In [35]:
# Guardar os três resultados de O2 e voltar a lê-los.
results_o2 = {
    "route_month": route_month,
    "reversal": reversal,
    "top": top
}

for name, result in results_o2.items():
    path = os.path.join(outFolder, "validacao_o2", name)

    result.write.mode("overwrite").parquet(path)
    saved = spark.read.parquet(path).select(result.columns)

    same = (
        result.exceptAll(saved).count() == 0
        and saved.exceptAll(result).count() == 0
    )

    print(name, "— conteúdo preservado:", same)

route_month — conteúdo preservado: True


reversal — conteúdo preservado: True


top — conteúdo preservado: True


### 6.9 Ler o plano físico

Procurar a agregação das viagens, a janela das participações e as duas junções com as zonas. Neste plano aparecem dois BroadcastHashJoin BuildRight e leitura da cache. Os Parquet sob InMemoryRelation mostram a origem da cache, não provam uma nova leitura.


In [36]:
route_month.explain("formatted")


== Physical Plan ==
AdaptiveSparkPlan (100)
+- Project (99)
   +- BroadcastHashJoin Inner BuildRight (98)
      :- Project (93)
      :  +- BroadcastHashJoin Inner BuildRight (92)
      :     :- Project (87)
      :     :  +- Filter (86)
      :     :     +- Window (85)
      :     :        +- Sort (84)
      :     :           +- Exchange (83)
      :     :              +- HashAggregate (82)
      :     :                 +- Exchange (81)
      :     :                    +- HashAggregate (80)
      :     :                       +- Filter (79)
      :     :                          +- InMemoryTableScan (1)
      :     :                                +- InMemoryRelation (2)
      :     :                                      +- * Project (78)
      :     :                                         +- * Filter (77)
      :     :                                            +- * Sample (76)
      :     :                                               +- Union (75)
      :     :                  

Validação na amostra de 10%: contagens, chaves, participações e posições passaram. O top do Bronx coincidiu com o cálculo direto; conferimos um exemplo de inversão e as três escritas deram True. A execução com 100% dos dados está em `validacao_completa.ipynb`.


## 7. Estado da validação

As verificações de M1, M2, M3, O1 e O2 passaram na execução guardada. O anexo seguinte confere arredondamento e estabilidade numérica. Os tempos de desempenho e as respetivas decisões estão no notebook de experiências.


## 8. Anexo — arredondamento e estabilidade

Guardamos os exemplos que motivaram as correções e os testes com outras distribuições dos mesmos dados. Não misturar estes testes com a medição da pipeline.


### 8.1 Exemplos de arredondamento

Comparamos double e decimal no grupo usado para investigar a tarifa. M2 e M3 usam agora a mesma precisão na média monetária.


In [42]:
brooklyn_group = trips_pu.filter(
    (sf.col("pickup_borough") == "Brooklyn")
    & (sf.col("day_of_week") == 2)
    & (sf.col("hour_of_day") == 23)
)

brooklyn_group.agg(
    sf.count("*").alias("viagens"),

    sf.avg("fare_amount").alias("media_double"),

    sf.round(
        sf.avg("fare_amount"), 2
    ).alias("double_arredondado"),

    sf.avg(
        sf.col("fare_amount").cast("decimal(20, 6)")
    ).alias("media_decimal"),

    sf.round(
        sf.avg(sf.col("fare_amount").cast("decimal(20, 6)")),
        2
    ).alias("decimal_arredondado")
).show(truncate=False)

+-------+------------------+------------------+-------------+-------------------+
|viagens|media_double      |double_arredondado|media_decimal|decimal_arredondado|
+-------+------------------+------------------+-------------+-------------------+
|100    |23.055000000000003|23.06             |23.0550000000|23.06              |
+-------+------------------+------------------+-------------+-------------------+



### 8.2 Comparar M2 recalculado com o ficheiro

Voltamos a calcular M2, escrevemos por borough e lemos o resultado. Nesta execução esperamos True, mesmo com o recálculo.


In [43]:
m2_path = os.path.join(outFolder, "validacao_m2")

# Guardar o resultado em pastas por borough
m2.write.mode("overwrite").partitionBy("pickup_borough").parquet(m2_path)

# Voltar a ler o que foi guardado
m2_saved = spark.read.parquet(m2_path)

print("Pastas por borough:")
print(sorted(
    name for name in os.listdir(m2_path)
    if name.startswith("pickup_borough=")
))

print("Linhas antes:", m2.count())
print("Linhas depois:", m2_saved.count())

same = (
    m2.exceptAll(m2_saved.select(m2.columns)).count() == 0
    and m2_saved.select(m2.columns).exceptAll(m2).count() == 0
)
print("Conteúdo preservado:", same)

Pastas por borough:
['pickup_borough=Bronx', 'pickup_borough=Brooklyn', 'pickup_borough=EWR', 'pickup_borough=Manhattan', 'pickup_borough=Queens', 'pickup_borough=Staten Island']


Linhas antes: 877
Linhas depois: 877


Conteúdo preservado: True


### 8.3 Confirmar ausência de diferenças

Executar depois de 8.2. As duas tabelas devem ficar vazias.


In [44]:
saved = m2_saved.select(m2.columns)

print("Linhas do cálculo atual que diferem do ficheiro:")
m2.exceptAll(saved).show(10, truncate=False)

print("Linhas do ficheiro que diferem do cálculo atual:")
saved.exceptAll(m2).show(10, truncate=False)

Linhas do cálculo atual que diferem do ficheiro:


+--------------+-----------+-----------+----------+----------------+--------------------------+---------------------+
|pickup_borough|day_of_week|hour_of_day|trip_count|mean_fare_amount|mean_trip_duration_minutes|mean_trip_distance_km|
+--------------+-----------+-----------+----------+----------------+--------------------------+---------------------+
+--------------+-----------+-----------+----------+----------------+--------------------------+---------------------+

Linhas do ficheiro que diferem do cálculo atual:


+--------------+-----------+-----------+----------+----------------+--------------------------+---------------------+
|pickup_borough|day_of_week|hour_of_day|trip_count|mean_fare_amount|mean_trip_duration_minutes|mean_trip_distance_km|
+--------------+-----------+-----------+----------+----------------+--------------------------+---------------------+
+--------------+-----------+-----------+----------+----------------+--------------------------+---------------------+



### 8.4 Estabilidade numérica

Repetimos os cálculos com os mesmos dados, usando a distribuição original, 12 e 24 partições. O reparticionamento é só um teste, não uma otimização proposta.

Contagens, chaves, posições e classificações devem coincidir exatamente. Nas estatísticas sem arredondar, usamos tolerância absoluta de 0,000000001 mais 0,000000000001 vezes o valor. Também verificamos separadamente os valores apresentados depois do arredondamento.


#### 8.4.1 Preparar as comparações

Usamos apenas tabelas agregadas pequenas no Pandas. As viagens ficam no Spark.

Nos resultados esperamos zero em missing_keys, exact_changes e outside_tolerance. max_difference mostra a maior diferença sem arredondar.


In [45]:
# Comparar tabelas pequenas pelas chaves, sem depender da ordem das linhas.
import pandas as pd

stability_results = []

def compare_tables(label, left, right, keys, exact, numeric):
    paired = left.merge(right, on=keys, how="outer", suffixes=("_a", "_b"),
                        indicator=True, validate="one_to_one")
    missing = int((paired["_merge"] != "both").sum())
    exact_changes = 0
    changed_columns = {}
    for column in exact:
        a, b = paired[column + "_a"], paired[column + "_b"]
        changed = ~(a.eq(b) | (a.isna() & b.isna()))
        exact_changes += int(changed.sum())
        if changed.any():
            changed_columns[column] = int(changed.sum())
            print("Diferenças em", column, paired.loc[changed, keys + [column + "_a", column + "_b"]].head(5).to_dict("records"))
    numeric_changes = 0
    max_difference = 0.0
    for column in numeric:
        a, b = paired[column + "_a"], paired[column + "_b"]
        difference = (a - b).abs()
        allowed = 1e-9 + 1e-12 * pd.concat([a.abs(), b.abs()], axis=1).max(axis=1)
        invalid = a.isna().ne(b.isna()) | (difference > allowed)
        numeric_changes += int(invalid.sum())
        if difference.notna().any():
            max_difference = max(max_difference, float(difference.max()))
    result = dict(test=label, missing_keys=missing, exact_changes=exact_changes,
                  outside_tolerance=numeric_changes, max_difference=max_difference, changed_columns=changed_columns)
    stability_results.append(result)
    print(result)
    assert missing == exact_changes == numeric_changes == 0, label

# Sem arredondar: distinguir uma pequena diferença de uma alteração relevante.
def raw_m2(frame):
    return frame.groupBy("pickup_borough", "day_of_week", "hour_of_day").agg(
        sf.avg("trip_distance_km").alias("distance")
    ).toPandas()

def raw_m3(frame):
    return frame.groupBy("PULocationID", "pickup_year", "pickup_month").agg(
        sf.avg("fare_amount").alias("fare_mean"),
        sf.stddev("fare_amount").alias("fare_std"),
        sf.avg("trip_distance_km").alias("distance_mean"),
        sf.stddev("trip_distance_km").alias("distance_std")
    ).toPandas()


#### 8.4.2 Conferir M2


In [46]:
# M2: contagens e valores apresentados têm de coincidir neste teste.
m2_keys = ["pickup_borough", "day_of_week", "hour_of_day"]
m2_reference = pl.m2_demand(trips_pu).toPandas()
m2_raw_reference = raw_m2(trips_pu)

for partitions in [12, 24]:
    distributed = trips_pu.repartition(partitions)
    compare_tables("M2 apresentado / " + str(partitions), m2_reference,
                   pl.m2_demand(distributed).toPandas(), m2_keys,
                   ["trip_count", "mean_fare_amount", "mean_trip_duration_minutes",
                    "mean_trip_distance_km"], [])
    compare_tables("M2 sem arredondar / " + str(partitions), m2_raw_reference,
                   raw_m2(distributed), m2_keys, [], ["distance"])


{'test': 'M2 apresentado / 12', 'missing_keys': 0, 'exact_changes': 0, 'outside_tolerance': 0, 'max_difference': 0.0, 'changed_columns': {}}


{'test': 'M2 sem arredondar / 12', 'missing_keys': 0, 'exact_changes': 0, 'outside_tolerance': 0, 'max_difference': 4.263256414560601e-14, 'changed_columns': {}}


{'test': 'M2 apresentado / 24', 'missing_keys': 0, 'exact_changes': 0, 'outside_tolerance': 0, 'max_difference': 0.0, 'changed_columns': {}}


{'test': 'M2 sem arredondar / 24', 'missing_keys': 0, 'exact_changes': 0, 'outside_tolerance': 0, 'max_difference': 1.1368683772161603e-13, 'changed_columns': {}}


#### 8.4.3 Conferir M3


In [47]:
# M3: conferir estatísticas e janelas, incluindo as classificações de anomalia.
m3_keys = ["PULocationID", "pickup_year", "pickup_month"]
m3_reference = pl.m3_zone_month(spark, trips_pu, zones, YEARS).toPandas()
m3_raw_reference = raw_m3(trips_pu)
m3_numbers = ["fare_mean", "fare_std", "distance_mean", "distance_std"]
m3_exact = ["pickup_borough", "pickup_zone", "trip_count", "rank_in_borough",
            "share_3m", "rolling_mean_3m", "rolling_std_3m", "volume_anomaly",
            "mean_fare_amount", "std_fare_amount", "mean_trip_distance_km",
            "std_trip_distance_km"]

for partitions in [12, 24]:
    distributed = trips_pu.repartition(partitions)
    compare_tables("M3 apresentado / " + str(partitions), m3_reference,
                   pl.m3_zone_month(spark, distributed, zones, YEARS).toPandas(),
                   m3_keys, m3_exact, [])
    compare_tables("M3 sem arredondar / " + str(partitions), m3_raw_reference,
                   raw_m3(distributed), m3_keys, [], m3_numbers)


{'test': 'M3 apresentado / 12', 'missing_keys': 0, 'exact_changes': 0, 'outside_tolerance': 0, 'max_difference': 0.0, 'changed_columns': {}}


{'test': 'M3 sem arredondar / 12', 'missing_keys': 0, 'exact_changes': 0, 'outside_tolerance': 0, 'max_difference': 3.637978807091713e-12, 'changed_columns': {}}


{'test': 'M3 apresentado / 24', 'missing_keys': 0, 'exact_changes': 0, 'outside_tolerance': 0, 'max_difference': 0.0, 'changed_columns': {}}


{'test': 'M3 sem arredondar / 24', 'missing_keys': 0, 'exact_changes': 0, 'outside_tolerance': 0, 'max_difference': 4.547473508864641e-12, 'changed_columns': {}}


#### 8.4.4 Conferir as anomalias de O1


In [48]:
# O1: comparar os limites e a classificação de cada viagem, não só os totais.
bucket_keys = ["PULocationID", "hour_of_day"]

def bucket_limits(frame):
    return (frame.withColumn("fare_per_km", sf.try_divide("fare_amount", "trip_distance_km"))
            .groupBy(*bucket_keys)
            .agg(sf.avg("fare_per_km").alias("mean"),
                 sf.stddev("fare_per_km").alias("std")))

reference_limits = bucket_limits(trips_pu)
reference_rows = reference_limits.collect()
reference_fixed = spark.createDataFrame(reference_rows, reference_limits.schema)
reference_pd = reference_fixed.toPandas()

for partitions in [12, 24]:
    alternative = bucket_limits(trips_pu.repartition(partitions))
    alternative_fixed = spark.createDataFrame(alternative.collect(), alternative.schema)
    compare_tables("O1 limites / " + str(partitions), reference_pd,
                   alternative_fixed.toPandas(), bucket_keys, [], ["mean", "std"])
    limits = reference_fixed.select(*bucket_keys,
        (sf.col("mean") + 3 * sf.col("std")).alias("limit_a")
    ).join(alternative_fixed.select(*bucket_keys,
        (sf.col("mean") + 3 * sf.col("std")).alias("limit_b")
    ), on=bucket_keys)
    classified = trips_pu.join(sf.broadcast(limits), on=bucket_keys).withColumn(
        "fare_per_km", sf.try_divide("fare_amount", "trip_distance_km")
    )
    a = sf.coalesce(sf.col("fare_per_km") > sf.col("limit_a"), sf.lit(False))
    b = sf.coalesce(sf.col("fare_per_km") > sf.col("limit_b"), sf.lit(False))
    changes = classified.agg(sf.sum((a != b).cast("long")).alias("changes")).first()["changes"]
    print("O1: classificações diferentes com", partitions, "partições:", changes)
    stability_results.append(dict(test="O1 classificações / " + str(partitions), changes=changes))
    assert changes == 0


{'test': 'O1 limites / 12', 'missing_keys': 0, 'exact_changes': 0, 'outside_tolerance': 0, 'max_difference': 9.094947017729282e-13, 'changed_columns': {}}


O1: classificações diferentes com 12 partições: 0


{'test': 'O1 limites / 24', 'missing_keys': 0, 'exact_changes': 0, 'outside_tolerance': 0, 'max_difference': 9.094947017729282e-13, 'changed_columns': {}}


O1: classificações diferentes com 24 partições: 0


#### 8.4.5 Guardar as verificações


In [49]:
# Guardar as verificações desta execução, separadas das medições de desempenho.
import json

os.makedirs(outFolder, exist_ok=True)
with open(os.path.join(outFolder, "estabilidade_numerica.json"), "w") as file:
    json.dump({"fraction": FRACTION, "years": YEARS, "checks": stability_results}, file, indent=2)
print("Testes de estabilidade concluídos para", FRACTION * 100, "% dos dados.")


Testes de estabilidade concluídos para 10.0 % dos dados.


#### 8.4.6 Conclusão

Os 12 testes passaram nesta execução: os valores apresentados de M2/M3 coincidiram e nenhuma classificação comparada mudou. As diferenças nas estatísticas sem arredondar ficaram dentro da tolerância.

Decisão: manter decimal nas médias monetárias de M2/M3 e na duração de M2; manter double nas restantes estatísticas. Estes resultados dizem respeito a estes dados e às distribuições testadas, não garantem igualdade em qualquer execução.


## 9. Fechar a sessão

Executar no fim para libertar a cache e os recursos do cluster.


In [50]:
clean.unpersist()
spark.stop()
